In [ ]:
import torch

def tri_to_ellipsoid(tri_v):

    def face_normals(face_vert, unit):
        edges_dist0 = face_vert[:, 1, :] - face_vert[:, 0, :]
        edges_dist1 = face_vert[:, 2, :] - face_vert[:, 0, :]
        face_normals = torch.cross(edges_dist0, edges_dist1, dim=-1)
        if unit:
            face_normals = torch.nn.functional.normalize(face_normals)
        return face_normals

    face_vert = torch.tensor(tri_v, dtype=torch.float32).view(1, 3, 3)
    n_face = 1
    v1v2 = face_vert[:, 1, :] - face_vert[:, 0, :]
    v1v2_len = torch.norm(v1v2, dim=-1, keepdim=True).clamp_min(1e-12)
    face_x = torch.div(v1v2, v1v2_len)
    face_normals = face_normals(face_vert, unit=True)
    face_y = torch.nn.functional.normalize(torch.cross(face_normals, face_x, dim=-1))
    rot_t2w = torch.stack([face_x, face_y, face_normals], dim=2)
    v1v2_xlen = v1v2_len.view(n_face)
    v1v3 = face_vert[:, 2, :] - face_vert[:, 0, :]


    A = torch.stack([(v1v3*face_x).sum(-1), (v1v3*face_y).sum(-1)], dim=-1)
    E = torch.stack([v1v2_xlen/2., v1v2_xlen * ((3**0.5)/2)], dim=-1)
    M_2d = torch.zeros((n_face, 2, 2))
    M_2d[:, 0, 0] = 1.
    M_2d[:, 0, 1] = torch.div((A[:, 0] - E[:, 0]), E[:, 1])
    M_2d[:, 1, 1] = torch.div(A[:, 1], E[:, 1])
    rad_base = 1. / (2.*(3**0.5))
    s_scalar = (v1v2_xlen * rad_base)  # s_scalar is std-deviation
    L = M_2d * s_scalar.view(-1, 1, 1)

    cov_matrix = torch.bmm(L, L.transpose(1, 2))
    eigenvalues, eigenvectors = torch.linalg.eigh(cov_matrix)
    scale2D = torch.sqrt(eigenvalues)
    _scaling_local = torch.cat([scale2D, torch.full_like(scale2D[:, :1], 1e-6)], dim=-1)

    _rotation_local = torch.zeros(n_face, 3, 3)
    _rotation_local[:, :2, :2] = eigenvectors
    _rotation_local[:,  2,  2] = torch.det(eigenvectors)  # important to ensure a rotation matrix

    _scaling = _scaling_local
    _rotation = torch.bmm(rot_t2w, _rotation_local)
    
    return _scaling[0], _rotation[0]

In [ ]:
import torch
import numpy as np
import plotly.graph_objects as go
from utils import plot_ellipsoid, rotation_matrix

tri_v = (np.random.rand(3, 3) - 0.5)/100
# a real case
tri_v = torch.tensor([[-0.3814,  0.1391,  0.5724],
                        [-0.3885,  0.1319,  0.6111],
                        [-0.3849,  0.1420,  0.5917]]).numpy()
tri_v = np.concatenate([tri_v, tri_v[0:1, :]], axis=0)
tri = go.Scatter3d(x=tri_v[:, 0], y=tri_v[:, 1], z=tri_v[:, 2], mode='lines+markers', 
                   marker=dict(size=5), line=dict(color='red', width=2))

# adaptive
face_vert = torch.tensor(tri_v[None, :3, :])

s, R = tri_to_ellipsoid(tri_v[:3, :])

ellipsoid = plot_ellipsoid(tri_v[:3].mean(axis=0), s.tolist(), 
                           R.numpy())

# Create the 3D surface plot
fig = go.Figure(data=[ellipsoid, tri])

fig.update_layout(
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    ),
    width=400,
    margin=dict(r=10, b=10, l=10, t=0)
)
fig.show()